# Suite XF — Transforms: models, procedures, application and certificates

A transform has a prototype, a procedure and a certificate ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)). Its
model is mbse-schemas data, installed by `Transforms.install` and defined by `Transforms.define`: its name, its
prototype (a pattern, which declares its parameters through `Parametrics`) and the name its procedure is registered
under. A procedure is a native function registered by name, called with the caller's visitors and the parameter
assignments, and that's that: `procedure(source, sync, target, assignments)`, each a `Visitors.OfPattern` or null.
`Transforms.apply` receives the caller's visitors, the transform and the assignments, and calls the procedure, and
that's that; the caller verifies, if at all, with its own call; then `Transforms.certify`, given the same, the
certificates' store, the history and who decided the parameters, writes the certificates ([CAP-075: steps linked to what they matched and wrote](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-075.md)), in the one transaction mbse-transforms opens: a history certificate,
the sequence of transform certificates and who decided each; a transform certificate per application, with its
parameter assignments; and a sub-certificate per argument present, its pattern instance and the transform before it
on the same history.

The tests play the caller: each makes its visitors within its own transactions, from frames it obtained from its
stores (`support.bear`, which makes a `support.Frame`, the caller's `Visitors.OfPattern`), and passes them in. The
fixture is a shelf: a singleton `shelf.Shelf` owning `shelf.Item`s by key, each with a `name` and a `label`; the
pattern `shelf.Unlabelled`, an item without a label, which declares the parameter `case`; and `shelf.Whole`, the
shelf. The transforms are defined in a store of their own, `transforms`, and the certificates written in another,
`certs`.

In [ ]:
import { Builders as B, Parametrics, Plain, Proxies, References } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";

import { Transforms } from "../src/index.js";

import { assert, bear, equal, raises } from "./support.js";

/** shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name and a label. */
function shelf(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("xf.shelf/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Shelf: S.OfObject, Item: S.OfObject, Holds: S.OfRelation })
    .spec([
      (_: any) => _.Shelf.name("shelf.Shelf").ref(true).singleton("shelf.TheShelf").adjacencies({
        items: (t: any) => t.relation([_.Holds]).me("shelf") }),
      (_: any) => _.Item.name("shelf.Item").ref(true).properties({ name: S.OfStr, label: S.OfStr }).adjacencies([
        (t: any) => t.name("shelf").relation([_.Holds]).me("item").owner(true)]),
      (_: any) => _.Holds.name("shelf.Holds").links([(t: any) => t.name("shelf"), (t: any) => t.name("item")])
        .properties({ key: S.OfStr }).uniques([["shelf", "key"], ["item"]]),
    ])]))]);
}

/** shelf.Unlabelled, an item without a label, declaring the parameter case; and shelf.Whole, the shelf. */
function patterns(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("xf.patterns/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Unlabelled: S.OfPattern, Whole: S.OfPattern })
    .spec([
      (_: any) => _.Unlabelled.name("shelf.Unlabelled").symbols({ i: txn.shelf.Item })
        .spec([(f: any) => f.i.label(null)])
        .parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          case: (d: any) => d.parameter((p: any) => p.description("upper or lower").for_value([S.OfStr])) }))]),
      (_: any) => _.Whole.name("shelf.Whole").symbols({ s: txn.shelf.Shelf }),
    ])]))]);
}

/** A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it. */
function model(...names: string[]): any {
  const store: any = new B.Store();
  store.transaction(Parametrics.install);
  store.transaction(shelf);
  store.transaction(patterns);
  store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ s: t.shelf.Shelf })]))]));
  for (const name of names) {
    store.transaction(() => {
      store.shelf.Shelf(store.singleton("shelf.TheShelf"), (s: any) => s.items({
        [name]: (e: any) => e.item((i: any) => i.name(name)) }));
    });
  }
  return store;
}

/** The frame of shelf.Unlabelled binding the item `name`, as `store.transaction` gives it back. */
function unlabelled(store: any, name: string): any {
  const [frame] = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ i: t.shelf.Item }).spec([(_: any) => _.i.name(name).label(null)])]))]));
  return frame;
}

/** The frame of shelf.Whole, as `store.transaction` gives it back. */
function whole(store: any): any {
  const held = store.registered("shelf.Whole");
  const [frame] = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([held]))]));
  return frame;
}

/** Each item on the shelf: its key, name and label. */
function items(store: any): unknown[][] {
  return store.transaction(() => [...store.singleton("shelf.TheShelf").items].map((e: any) =>
    [e.key.get(), e.item.name.get(), e.item.label.get()]));
}

/** The parameter assignments a transform certificate records, read through the visitor protocol. */
function assignments_of(certificate: any): Record<string, unknown> {
  const held: Record<string, unknown> = {};
  const each = (entries: any) => {
    for (const [name, value] of entries) {
      value.as_scalar((s: any) => {
        held[name] = s.get();
        return s;
      });
    }
    return entries;
  };
  certificate.assignments.as_value((a: any) => a.as_container((c: any) => c.as_entries(each)));
  return held;
}

/** A value object a certificate holds, as plain data. */
function plain(store: any, value: any): any {
  const held: any[] = [];
  value.as_object((o: any) => {
    held.push(Plain.ToPlain(store)(o.as_schema(), o));
    return o;
  });
  return held[0];
}

/** The number of transform certificates in a history. */
function steps(history: string): number {
  return certs.transaction(() => {
    try {
      return [...certs.singleton("Transforms.TheHistories").histories.at(history).history.steps].length;
    } catch { // at throws only KeyError: no history of that name
      return 0;
    }
  });
}

const transforms: any = model(); // the transforms' store: the shelf's schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install);
for (const define of [
  (t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled),
  (t: any) => Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled),
]) {
  transforms.transaction(define);
}
const certs: any = new B.Store(); // the certificates' store, as the caller gives it
certs.transaction(Transforms.install);
const procedures = new Transforms.Procedures();

/** The caller's application: within a transaction of the transforms' store and of each argument's store, the
 * transform's visitor, and each argument's visitor, made from the frame given, `[store, pattern, frame]`, applied and
 * then certified; the certificate's reference. */
function application(name: string, assignments: Transforms.Assignments, history: string, by: string,
  given: Record<string, [any, string, any]>): any {
  const stores = [...new Set([transforms, ...Object.values(given).map((a) => a[0])])];
  const within = (k: number): any => k < stores.length ? stores[k].transaction(() => within(k + 1)) : (() => {
    const transform = transforms.singleton("Transforms.TheCatalog").transforms.at(name).transform;
    const visitors = Object.fromEntries(Object.entries(given).map(([argument, [store, pattern, frame]]) =>
      [argument, bear(store, pattern, frame)]));
    Transforms.apply(transform, assignments, procedures, visitors);
    return Transforms.certify(certs, transform, assignments, history, by, visitors);
  })();
  return within(0);
}

/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const item = sync.i;
  const name = String(item.name);
  Proxies.store_of(item).shelf.Item(item, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase()));
}

procedures.register("label", label);

/** Puts on the target's shelf an item named as the source's, in the case its parameter assignment gives. */
function copy(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  let name = String(source.i.name);
  name = assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase();
  Proxies.store_of(target.s).shelf.Shelf(target.s, (s: any) => s.items({ [name]: (e: any) => e.item((i: any) => i.name(name)) }));
}

procedures.register("copy", copy);

<a id="XF-01"></a>
## XF-01 · A transform's anatomy: its name, prototype, parameters and the name its procedure is registered under, defined as data

In [ ]:
{
  const Unlabelled = transforms.registered("shelf.Unlabelled");
  transforms.transaction(() => {
    const entry = transforms.singleton("Transforms.TheCatalog").transforms.at("Label");
    const Label = entry.transform;
    assert(equal([String(entry.name), String(Label.procedure)], ["Label", "label"]));
    assert(equal([...Label.prototype].map((e: any) => e.pattern), [Unlabelled]));
    const [held, ...more] = [...new B.OfPattern.Builder(Unlabelled, false, transforms).parameters]; // its parameters: its prototype's
    assert(more.length === 0 && equal([...held.params.parameters].map((e: any) => String(e.name)), ["case"]));
  });
  assert(equal(Parametrics.validate(transforms, Unlabelled), []));
  assert(equal([...transforms.registered("Transforms.Transform").properties.keys()].sort(), ["procedure"])); // nothing else
  const snapshot = Plain.ToPlain(transforms).Snapshot();
  transforms.transaction(Transforms.install); // installed again, defined again: found, nothing written
  transforms.transaction((t: any) => Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled));
  assert(equal(Plain.ToPlain(transforms).Snapshot(), snapshot));
  assert(transforms.registered("Transforms.Transform").ref && certs.registered("Transforms.TransformCertificate").ref);
}

<a id="XF-02"></a>
## XF-02 · Procedures are registered by name, each once; an unknown name is refused before anything runs

In [ ]:
{
  assert(procedures.has("label") && !procedures.has("ghost"));
  raises(ValueError, "a procedure is registered as 'label' already", () => procedures.register("label", label));
  const other = new Transforms.Procedures(); // a registry is a value: none is global
  assert(!other.has("label") && other.register("label", label) === label);
  transforms.transaction((t: any) => Transforms.define(t, "Ghost", "ghost", t.shelf.Unlabelled));
  const data = model("bolt");
  raises(LookupError, "no procedure is registered as 'ghost'", () =>
    application("Ghost", { case: "upper" }, "run", "caller", { sync: [data, "shelf.Unlabelled", unlabelled(data, "bolt")] }));
  assert(steps("run") === 0 && equal(items(data), [["bolt", "bolt", null]]));
}

<a id="XF-03"></a>
## XF-03 · A procedure is called with the caller's visitors and the parameter assignments, and that's that: each argument absent is null

In [ ]:
{
  const seen: unknown[][] = [];
  procedures.register("inspect", (source, sync, target, assignments) => seen.push(
    [source, sync, target, { ...assignments }, String(source.i.name), source.as_schema().name]));

  /** Tries to change its parameter assignments. */
  const frozen = (source: any, sync: any, target: any, assignments: Transforms.Assignments): void => {
    (assignments as Record<string, unknown>).case = "lower";
  };

  procedures.register("frozen", frozen);

  transforms.transaction((t: any) => Transforms.define(t, "Inspect", "inspect", t.shelf.Unlabelled));
  transforms.transaction((t: any) => Transforms.define(t, "Frozen", "frozen", t.shelf.Unlabelled));
  const data = model("bolt");
  const frame = unlabelled(data, "bolt");
  transforms.transaction(() => data.transaction(() => {
    const source = bear(data, "shelf.Unlabelled", frame);
    Transforms.apply(transforms.singleton("Transforms.TheCatalog").transforms.at("Inspect").transform,
      { case: "upper" }, procedures, { source });
    assert(seen.length === 1 && seen[0]![0] === source // the very visitor given
      && equal(seen[0]!.slice(1), [null, null, { case: "upper" }, "bolt", "shelf.Unlabelled"]));
    raises(TypeError, "read only", () => // read-only: what the certificate records is what the procedure was given
      Transforms.apply(transforms.singleton("Transforms.TheCatalog").transforms.at("Frozen").transform,
        { case: "upper" }, procedures, { source }));
  }));
  assert(steps("inspect") === 0); // apply writes nothing: certify does
}

<a id="XF-04"></a>
## XF-04 · The arguments are the caller's: the source read, the sync edited in place and the target built, each a visitor bearing its own pattern frame, in the caller's transactions, which commit or roll back what the procedure wrote

In [ ]:
{
  procedures.register("tag", (source, sync, target, assignments) => {
    label(source, sync, target, assignments);
    copy(source, sync, target, assignments);
  });
  transforms.transaction((t: any) => Transforms.define(t, "Tag", "tag", t.shelf.Unlabelled));
  const [source, sync, target] = [model("bolt", "nut"), model("bolt", "nut"), model()]; // the copies are the caller's to make
  application("Tag", { case: "upper" }, "arguments", "caller", { source: [source, "shelf.Unlabelled", unlabelled(source, "bolt")],
    sync: [sync, "shelf.Unlabelled", unlabelled(sync, "bolt")], target: [target, "shelf.Whole", whole(target)] });
  assert(equal(items(source), [["bolt", "bolt", null], ["nut", "nut", null]])); // read
  assert(equal(items(sync), [["bolt", "bolt", "BOLT"], ["nut", "nut", null]])); // edited in place
  assert(equal(items(target), [["BOLT", "BOLT", null]])); // built

  const nut = unlabelled(sync, "nut");
  raises(Error, "the caller rolls back", () => // neither opens a transaction on the visitors' stores
    transforms.transaction(() => sync.transaction(() => {
      const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
      const given = bear(sync, "shelf.Unlabelled", nut);
      Transforms.apply(Label, { case: "lower" }, procedures, { sync: given });
      Transforms.certify(certs, Label, { case: "lower" }, "arguments", "caller", { sync: given });
      throw new Error("the caller rolls back");
    })));
  assert(equal(items(sync)[1], ["nut", "nut", null]) && steps("arguments") === 2); // rolled back by the caller; certified
}

<a id="XF-05"></a>
## XF-05 · Parameters: the transform certificate records the parameter assignments, and the history certificate who decided them, as the caller says

In [ ]:
{
  const data = model("bolt", "nut");
  const first = application("Label", { case: "upper" }, "run", "caller", { sync: [data, "shelf.Unlabelled", unlabelled(data, "bolt")] });
  application("Label", { case: "lower" }, "run", "policy", { sync: [data, "shelf.Unlabelled", unlabelled(data, "nut")] });
  assert(equal(items(data), [["bolt", "bolt", "BOLT"], ["nut", "nut", "nut"]]));
  certs.transaction(() => {
    const history = certs.singleton("Transforms.TheHistories").histories.at("run").history;
    assert(equal([...history.steps].map((e: any) => [Number(e.index), e.step.assignments.get()]), [[0, null], [1, null]])); // a container: not a scalar
    assert(equal([...history.steps].map((e: any) => assignments_of(e.step)), [{ case: "upper" }, { case: "lower" }]));
    assert(equal([...history.decisions].map((d: any) => String(d.by)), ["caller", "policy"])); // provenance: the history's, by step
    assert(equal([...history.decisions].map((d: any) => d.step), [...history.steps].map((e: any) => e.step)));
  });
  assert(equal(Plain.ToPlain(certs)(first.as_schema(), first).get("path").get("next").get("entry").get("next").get("entry")
    .get("properties"), new Map([["index", 0n]])));
}

<a id="XF-06"></a>
## XF-06 · Certificates are written and linked: the history's sequence, prev and next, the transform and its prototype, a sub-certificate for each argument present holding the pattern instance its visitor bore, and the chain of each pattern instance; all of it data, written and read back as any

In [ ]:
{
  const [source, target] = [model("bolt"), model()];
  const [bolt, shelf_] = [unlabelled(source, "bolt"), whole(target)];
  for (const caseOf of ["upper", "lower"]) { // the same pattern instances, twice
    application("Copy", { case: caseOf }, "chain", "caller", { source: [source, "shelf.Unlabelled", bolt],
      target: [target, "shelf.Whole", shelf_] });
  }
  assert(equal(items(target), [["BOLT", "BOLT", null], ["bolt", "bolt", null]]));
  const expected_transform = transforms.transaction(() => {
    const Copy = transforms.singleton("Transforms.TheCatalog").transforms.at("Copy").transform;
    const copy_ref = References.of(transforms, Copy);
    return Plain.ToPlain(transforms)(copy_ref.as_schema(), copy_ref);
  });
  const pattern = References.of(source, source.registered("shelf.Unlabelled"));
  const expected_pattern = Plain.ToPlain(source)(pattern.as_schema(), pattern);
  const instance = References.of(source, new Map([["i", bolt.get("i")]])); // the frame the visitor bore: its symbols' bindings
  const expected_frame = Plain.ToPlain(source)(instance.as_schema(), instance);
  certs.transaction(() => {
    const history = certs.singleton("Transforms.TheHistories").histories.at("chain").history;
    const [one, two] = [...history.steps].map((e: any) => e.step);
    assert(equal([...one.next].map((e: any) => e.after), [two]) && equal([...two.prev].map((e: any) => e.before), [one]));
    assert([...one.prev].length === 0 && [...two.next].length === 0);
    assert(equal(plain(certs, one.transform), expected_transform)); // the transform, in its own store, by reference
    assert(equal(plain(certs, one.prototype), expected_pattern));
    assert(equal([...one.frames].map((e: any) => String(e.argument)), ["source", "target"])); // whichever are present
    const [first_source, first_target] = [...one.frames].map((e: any) => e.frame);
    const [second_source, second_target] = [...two.frames].map((e: any) => e.frame);
    assert(equal(plain(certs, first_source.frame), expected_frame)); // the pattern instance: the frame, by reference
    assert(equal(plain(certs, first_target.pattern).get("path").get("next").get("entry").get("next").get("entry").get("properties"),
      new Map([["name", "Whole"]])));
    assert([...first_source.preceding].length === 0 && equal([...second_source.preceding].map((e: any) => e.step), [one]));
    assert(equal([...second_target.preceding].map((e: any) => e.step), [one]));
  });

  const log = Plain.ToPlain(certs).Snapshot(); // the certificates are data: written and read back with no code of their own
  const back: any = new B.Store();
  Plain.FromPlain(back).Snapshot(log);
  back.transaction(() => {
    const [one, two] = [...back.singleton("Transforms.TheHistories").histories.at("chain").history.steps].map((e: any) => e.step);
    assert(equal([assignments_of(one), assignments_of(two)], [{ case: "upper" }, { case: "lower" }]));
    assert(equal([...two.prev].map((e: any) => e.before), [one]) && equal(plain(back, one.prototype), expected_pattern));
  });
}

<a id="XF-07"></a>
## XF-07 · Verification is the caller's own call, between apply and certify, with the same visitors (the prototype falsified, or a metric decreased); where it fails, the caller does not certify, and no certificate is written

In [ ]:
{
  procedures.register("idle", () => null);
  transforms.transaction((t: any) => Transforms.define(t, "Idle", "idle", t.shelf.Unlabelled));

  /** The caller's falsification: the sync's item, a frame of shelf.Unlabelled, has a label now. */
  const falsified = (sync: any): boolean => sync.i.label.get() !== null;

  let data = model("bolt", "nut");
  const bolt = unlabelled(data, "bolt");
  transforms.transaction(() => data.transaction(() => { // the caller applies, verifies, and certifies only what holds
    const catalog = transforms.singleton("Transforms.TheCatalog").transforms;
    for (const name of ["Idle", "Label"]) {
      const [transform, sync] = [catalog.at(name).transform, bear(data, "shelf.Unlabelled", bolt)];
      Transforms.apply(transform, { case: "upper" }, procedures, { sync });
      if (falsified(sync)) { // Idle's fails: not certified
        Transforms.certify(certs, transform, { case: "upper" }, "verify", "caller", { sync });
      }
    }
  }));
  assert(steps("verify") === 1 && equal(items(data)[0], ["bolt", "bolt", "BOLT"]));
  application("Idle", { case: "upper" }, "verify", "caller", { sync: [data, "shelf.Unlabelled", unlabelled(data, "nut")] }); // unverified
  assert(steps("verify") === 2);

  /** How many items on the sync's shelf have no label: a metric. */
  const unlabelled_count = (sync: any): number =>
    [...Proxies.store_of(sync.i).singleton("shelf.TheShelf").items].filter((e: any) => e.item.label.get() === null).length;

  data = model("nut");
  const nut = unlabelled(data, "nut");
  transforms.transaction(() => data.transaction(() => { // a metric: the caller computes it before, and compares after
    const catalog = transforms.singleton("Transforms.TheCatalog").transforms;
    const sync = bear(data, "shelf.Unlabelled", nut);
    for (const name of ["Idle", "Label"]) {
      const before = unlabelled_count(sync);
      Transforms.apply(catalog.at(name).transform, { case: "lower" }, procedures, { sync });
      if (unlabelled_count(sync) < before) { // Idle's has not decreased: not certified
        Transforms.certify(certs, catalog.at(name).transform, { case: "lower" }, "metric", "caller", { sync });
      }
    }
  }));
  assert(steps("metric") === 1 && equal(items(data), [["nut", "nut", "nut"]]));
  certs.transaction(() => { // what was certified is Label's application
    const histories = certs.singleton("Transforms.TheHistories").histories;
    assert(equal(["verify", "metric"].map((h) => plain(certs, histories.at(h).history.steps.at(0n).step.transform)
      .get("path").get("next").get("entry").get("properties")), [new Map([["name", "Label"]]), new Map([["name", "Label"]])]));
  });
}

<a id="XF-08"></a>
## XF-08 · What an application refuses: a transform with no prototype, and a procedure that raises writes nothing

In [ ]:
{
  transforms.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ catalog: t.Transforms.Catalog, transform: t.Transforms.Transform }).spec([
      (_: any) => _.catalog.transforms({ Bare: (e: any) => e.transform(_.transform) }),
      (_: any) => _.transform.procedure("label")])]))]));
  const data = model("bolt");
  raises(ValueError, "transform 'Bare' has no prototype", () =>
    application("Bare", { case: "upper" }, "refused", "caller", { sync: [data, "shelf.Unlabelled", unlabelled(data, "bolt")] }));

  const broken = (source: any, sync: any, target: any, assignments: Transforms.Assignments): void => {
    label(source, sync, target, assignments);
    throw new Error("the procedure failed");
  };

  procedures.register("broken", broken);
  transforms.transaction((t: any) => Transforms.define(t, "Broken", "broken", t.shelf.Unlabelled));
  raises(Error, "the procedure failed", () =>
    application("Broken", { case: "upper" }, "refused", "caller", { sync: [data, "shelf.Unlabelled", unlabelled(data, "bolt")] }));
  assert(equal(items(data), [["bolt", "bolt", null]]) && steps("refused") === 0); // the caller's transaction rolled back
}